# Classification Partie 2 - Demandes de prêt immobilier aux États-Unis

## Import des librairies :

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier

## Pré-traitement sur les données :

##### Chargement des données d'entrainement et de test :

In [2]:
data_train = pd.read_csv("train.csv")
data_test = pd.read_csv("test.csv")

Ici, on choisit de laisser les NaN dans les deux datasets car on se concentre uniquement sur le modèle de gradient boosting qui les traite tout seul.

### Variables continues :

Rien à changer

### Variables catégorielles :

On traite les variables catégorielles pour qu'elles soient toutes de type float ou int :

##### Variables catégorielles ordinales :

Pour `applicant_age` et `co-applicant_age` :

In [3]:
ordre_age = {"<25": 25, "25-34": 30, "35-44": 40, "45-54": 50,
         "55-64": 60, "65-74": 70, ">74": 74}

data_train["applicant_age"] = data_train["applicant_age"].map(ordre_age)
data_train["co-applicant_age"] = data_train["co-applicant_age"].map(ordre_age)
data_test["applicant_age"] = data_test["applicant_age"].map(ordre_age)
data_test["co-applicant_age"] = data_test["co-applicant_age"].map(ordre_age)

Pour `debt_to_income_ratio` :

In [4]:
ordre_debt = {"<20%": 20, "20%-<30%": 25, "30%-<36%": 33, "36": 36, "37": 37, "38": 38,
              "39": 39, "40": 40, "41": 41, "42": 42, "43": 43, "44": 44, "45": 45, "46": 46,
              "47": 47, "48": 48, "49": 49, "50%-60%": 55, ">60%": 60}

data_train["debt_to_income_ratio"] = data_train["debt_to_income_ratio"].map(ordre_debt)
data_test["debt_to_income_ratio"] = data_test["debt_to_income_ratio"].map(ordre_debt)

##### Variables catégorielles nominales :

In [5]:
cols_nom = [
    "state_code", "conforming_loan_limit",
    "derived_loan_product_type", "derived_dwelling_category",
    "loan_type", "lien_status", "preapproval", "reverse_mortgage",
    "open-end_line_of_credit", "business_or_commercial_purpose",
    "negative_amortization", "interest_only_payment", "balloon_payment",
    "other_nonamortizing_features", "construction_method", "occupancy_type",
    "manufactured_home_secured_property_type",
    "manufactured_home_land_property_interest", "total_units",
    "applicant_credit_score_type", "co-applicant_credit_score_type",
    "derived_ethnicity", "derived_race", "derived_sex",
    "applicant_ethnicity-1", "co-applicant_ethnicity-1",
    "applicant_ethnicity_observed", "co-applicant_ethnicity_observed",
    "applicant_race-1", "co-applicant_race-1",
    "applicant_race_observed", "co-applicant_race_observed",
    "applicant_sex", "co-applicant_sex",
    "applicant_sex_observed", "co-applicant_sex_observed",
    "submission_of_application", "initially_payable_to_institution",
    "aus-1",
]

enc_nom = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
# unknown_value=-1 : catégorie vue dans le test mais pas dans le train
# fit sur le train uniquement, puis transform sur le test (évite la fuite de données)
data_train[cols_nom] = enc_nom.fit_transform(data_train[cols_nom])
data_test[cols_nom] = enc_nom.transform(data_test[cols_nom])

### Feature engineering (ajout de variables ratios) : 

In [6]:
def add_features(df):
    df = df.copy()
    inc = df["income"].where(df["income"] > 0) * 1000      # income est en milliers de $
    
    df["loan_to_income"]   = df["loan_amount"] / inc
    df["income_vs_msa"]    = inc / df["ffiec_msa_md_median_family_income"]
    df["loan_to_value_calc"] = df["loan_amount"] / df["property_value"]
    df["implied_value"]    = df["loan_amount"] / (df["loan_to_value_ratio"] / 100)
    df["value_gap"]        = df["implied_value"] / df["property_value"]
    df["monthly_principal"] = df["loan_amount"] / df["loan_term"]      
    df["pti_proxy"]        = df["monthly_principal"] / (inc / 12)      
    df["debt_amount"]      = df["debt_to_income_ratio"] * inc / 100    
    df["residual_income"]  = inc - df["loan_amount"] / (df["loan_term"] / 12)
    
    df["n_missing"] = df.isna().sum(axis=1)
    return df.replace([np.inf, -np.inf], np.nan)

data_train_feat = add_features(data_train)
data_test_feat = add_features(data_test)

##### On sépare les covariables de la variable cible dans les données d'entraînement :

In [7]:
X_train = data_train_feat.drop(columns=["accepted", "id"])
Y_train = data_train_feat["accepted"]
X_test = data_test_feat.drop(columns=["id"])

## Gradient Boosting :

##### Recherche d'hyperparamètres et entraînement :

Optimisation de `max_iter`, `learning_rate`, `max_depth`, `max_leaf_nodes`, `min_samples_leaf` et `l2_regularization` :

In [ ]:
params = {
    "max_iter": [200, 400, 600, 1000],
    "learning_rate": [0.02, 0.05, 0.1],
    "max_depth": [None, 4, 6, 8],
    "max_leaf_nodes": [15, 31, 63],
    "min_samples_leaf": [10, 20, 50, 100],
    "l2_regularization": [0, 0.1, 1, 10],
}

search_hgbc = RandomizedSearchCV(
    HistGradientBoostingClassifier(random_state=42),
    params, n_iter=20,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1, random_state=42,
)
search_hgbc.fit(X_train, Y_train)

print("Meilleurs paramètres :", search_hgbc.best_params_)
print(f"AUC CV : {search_hgbc.best_score_:.4f}")

##### Prédictions :

In [10]:
best_hgbc = search_hgbc.best_estimator_
y_proba = best_hgbc.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    "id": data_test["id"],
    "accepted": y_proba
})
submission.to_csv("classification_submission_gradient_boosting_NaN_addfeat.csv", index=False)